In [9]:
'''
Q1. Combine the datasets into a single master dataset. The combined data dataset should contain all the attributes found in all files. 
'''

"""
Task Name: HDB Raw Data Merge into a Single Dataset
Created by: ILANGO RAMESH
Created date: 06-Apr-2026
Notebook Name: HDB_Resale_ETL_Pipeline

Purpose:
- Merge 3 HDB resale datasets
- Keep raw month format (YYYY-MM) unchanged
- Standardize schema across all files
- Filter data for period (2012-01 to 2016-12)
- Produce a clean and structured output dataset
"""
# =========================
# IMPORT PYTHON LIBRARIES
# =========================

import os
import pandas as pd
import hashlib

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, to_date, count, when, isnan
from IPython.display import display
from datetime import datetime
from dotenv import load_dotenv


# =========================
# SPARK SESSION
# =========================
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("FixWindows") \
    .config("spark.hadoop.io.nativeio", "false") \
    .config("spark.sql.warehouse.dir", "file:///C:/temp") \
    .getOrCreate()
print("Spark Version:", spark.version)


# =========================
# RAW DATA FILE PATHS
# =========================
from dotenv import load_dotenv
import os

try:
    load_dotenv("hdb_resale.env")

    # Input Files (raw data)
    HDB_Resale_Transaction_Jan2012_Feb2012 = os.getenv("RAW_FILE_PATH_2000_FEB2012")
    HDB_Resale_Transaction_Mar2012_Dec2012 = os.getenv("RAW_FILE_PATH_MAR2012_DEC2014")
    HDB_Resale_Transaction_Jan2015_Dec2016 = os.getenv("RAW_FILE_PATH_JAN2015_DEC2016")
    
    # Validation
    if not HDB_Resale_Transaction_Jan2012_Feb2012 or not HDB_Resale_Transaction_Mar2012_Dec2012 or not HDB_Resale_Transaction_Jan2015_Dec2016:
        raise ValueError("One or more environment variables are missing!")

    # Output File (Merged all raw data files)
    Q1_Output_Path = os.getenv("Q1_OUTPUT_PATH")

    # Q2 : Output File  - DataFrame for profiling and processing
    Q2_Sliver_Output_Path  = os.getenv("Q2_OUTPUT_PATH")

    #Q3 : Output File  - HDB Data Validation Rules
    Q3_Valid_Dataset_Output_Path = os.getenv("Q3_VALID_OUTPUT_PATH")
    Q3_Invalid_Dataset_Output_Path = os.getenv("Q3_INVALID_OUTPUT_PATH")

    #Q4 : HDB Lease Recalculation
    Q4_Lease_Output_Path = os.getenv("Q4_LEASE_OUTPUT_PATH")

    # Q5 : HDB Composite Key Deduplication
    Q5_Compo_Dedup_Unique_Dataset_Output_Path = os.getenv("Q5_COMPOSITE_DEDUPLICATION_UNIQUE_OUTPUT_PATH")
    Q5_Compo_Dedup_Failed_Dataset_Output_Path = os.getenv("Q5_COMPOSITE_DEDUPLICATION_FAILED_OUTPUT_PATH")

    # Q6 : HDB Resale Price Anomaly Detection
    Q6_Price_Anomaly_Detection_Output_Path = os.getenv("Q6_PRICE_ANOMALY_DETECTION_OUTPUT_PATH")
    Q6_Price_Normal_Detection_Output_Path = os.getenv("Q6_PRICE_NORMAL_DETECTION_OUTPUT_PATH")
    
    # Q7 : Data Transformation    
    Q7_hdb_data_transformed_Output_Path = os.getenv("Q7_HDB_DATA_TRANSFORMED_OUTPUT_PATH")
    Q7_hdb_data_failed_Output_Path = os.getenv("Q7_HDB_DATA_FAILED_OUTPUT_PATH")
    Q7_hdb_data_hashed_Output_Path = os.getenv("Q7_HDB_DATA_HASHED_OUTPUT_PATH")
    
except Exception as e:
    print("Error while loading environment file.")
    print(e)
    raise
    
# ==============================
# LOAD DATA FROM RAW DATA PATHS
# ==============================
try:
    HDB_Resale_Transaction_Jan2012_Feb2012_df = spark.read.option("header", True).option("inferSchema", True).csv(HDB_Resale_Transaction_Jan2012_Feb2012)
    HDB_Resale_Transaction_Mar2012_Dec2012_df = spark.read.option("header", True).option("inferSchema", True).csv(HDB_Resale_Transaction_Mar2012_Dec2012)
    HDB_Resale_Transaction_Jan2015_Dec2016_df = spark.read.option("header", True).option("inferSchema", True).csv(HDB_Resale_Transaction_Jan2015_Dec2016)
except Exception as e:
    print("Error while reading CSV files")
    print(e)
    raise
    
# =========================
# MONTH FORMAT FIX (YYYY-MM)
# =========================
try:
    HDB_Resale_Transaction_Jan2012_Feb2012_df = HDB_Resale_Transaction_Jan2012_Feb2012_df.withColumn("month", date_format(to_date(col("month")), "yyyy-MM"))
    HDB_Resale_Transaction_Mar2012_Dec2012_df = HDB_Resale_Transaction_Mar2012_Dec2012_df.withColumn("month", date_format(to_date(col("month")), "yyyy-MM"))
    HDB_Resale_Transaction_Jan2015_Dec2016_df = HDB_Resale_Transaction_Jan2015_Dec2016_df.withColumn("month", date_format(to_date(col("month")), "yyyy-MM"))
except Exception as e:
    print("Error during month formatting")
    print(e)
    raise
    
# ==============================
# FILTER FILE 1 (JAN & FEB 2012)
# ==============================
try:
    HDB_Resale_Transaction_Jan2012_Feb2012_df = HDB_Resale_Transaction_Jan2012_Feb2012_df.filter((col("month") == "2012-01") | (col("month") == "2012-02"))
except Exception as e:
    print("Error during hdb sales data filtering")
    print(e)
    raise
    
# ================
# MERGE ALL FILES
# ================
hdb_resale_transaction_combined_df = HDB_Resale_Transaction_Jan2012_Feb2012_df \
        .unionByName(HDB_Resale_Transaction_Mar2012_Dec2012_df, allowMissingColumns=True) \
        .unionByName(HDB_Resale_Transaction_Jan2015_Dec2016_df, allowMissingColumns=True)

# =============
# FINAL SCHEMA
# =============
master_df = hdb_resale_transaction_combined_df.select(
    "month","town","flat_type","block","street_name","storey_range","floor_area_sqm",
    "flat_model","lease_commence_date","remaining_lease","resale_price")

#print("Final Dataframe Schema :" )
#master_df.printSchema()

# =========================
# SAVE OUTPUT
# =========================

try:    
    master_df.toPandas().to_csv(Q1_Output_Path,index=False)
    total_count = master_df.count()
    print(f"Total Records Count: {total_count}")
    
    print("The files have been merged successfully.")
    print("Output saved at:", Q1_Output_Path)
   
    # Display dataset sample for validation
    print("\n===== MERGED RAW DATA SAMPLE =====")
    display(master_df.head(5))

except Exception as e:
    print("Error while saving output file")
    print(e)
    raise

Spark Version: 3.5.5
Total Records Count: 92544
The files have been merged successfully.
Output saved at: C:/Users/ilang/HDB/hdb_resale/Q1_HDB_Bronze_Level_Data/Master_Dataset.csv

===== MERGED RAW DATA SAMPLE =====


[Row(month='2012-01', town='ANG MO KIO', flat_type='2 ROOM', block='406', street_name='ANG MO KIO AVE 10', storey_range='01 TO 03', floor_area_sqm=44.0, flat_model='Improved', lease_commence_date=1979, remaining_lease=None, resale_price=257800.0),
 Row(month='2012-01', town='ANG MO KIO', flat_type='2 ROOM', block='314', street_name='ANG MO KIO AVE 3', storey_range='07 TO 09', floor_area_sqm=44.0, flat_model='Improved', lease_commence_date=1978, remaining_lease=None, resale_price=263000.0),
 Row(month='2012-01', town='ANG MO KIO', flat_type='2 ROOM', block='314', street_name='ANG MO KIO AVE 3', storey_range='10 TO 12', floor_area_sqm=44.0, flat_model='Improved', lease_commence_date=1978, remaining_lease=None, resale_price=275000.0),
 Row(month='2012-01', town='ANG MO KIO', flat_type='2 ROOM', block='170', street_name='ANG MO KIO AVE 4', storey_range='01 TO 03', floor_area_sqm=45.0, flat_model='Improved', lease_commence_date=1986, remaining_lease=None, resale_price=260000.0),
 Row(month=

In [2]:
'''
Q2. Perform Data Profiling on the dataset. You may code your own profiling rules or leverage on open-source data quality frameworks. 
'''

"""
Task Name: HDB Resale Data - Silver Layer Processing (Data Profiling)

Created by: ILANGO RAMESH
Created date: 06-Apr-2026

Purpose:
Perform exploratory data profiling on the merged HDB resale dataset, including:
 - Basic data overview and structure validation
 - Missing value analysis and profiling
 - Duplicate record detection
 - Data type consistency checks
 - Unique value (cardinality) analysis
 - Descriptive statistical profiling
 - Silver layer data cleaning and standardization
 - Outlier detection
 - Categorical profiling
 - Numeric consistency checks
 - Date validation
 - Extended categorical profiling
 - Correlation analysis
 - Outlier reporting
 - Final dataset validation and readiness check
"""

# =========================================================
# 1. LOAD DATA
# =========================================================
# Convert Spark DataFrame (master_df) into Pandas DataFrame for profiling and processing
try:
    #silver_master_df =pd.read_csv(Q1_Output_Path)  # master_df.toPandas()   
    silver_master_df = master_df.toPandas()
except Exception as e:
    print("Error loading dataset")
    print(e)
    raise

print("===== SILVER LAYER DATA PROFILING STARTED =====")

# =========================================================
# 2. BASIC DATA OVERVIEW
# =========================================================
# Understand dataset structure, size, and sample records
print("\n===== 1. BASIC DATA OVERVIEW =====")
print("\nTotal Records:", len(silver_master_df))

print("\n ---- Dataset Info: ---- ")
print(silver_master_df.info())

print("\n ---- SAMPLE DATA ---- ")
display(silver_master_df.head(5))

# =========================================================
# 3. MISSING VALUE PROFILING
# =========================================================
# Identify missing values count and percentage per column
try:
    missing_count = silver_master_df.isnull().sum()
    missing_percent = (missing_count / len(silver_master_df)) * 100
    
    missing_profile = pd.DataFrame({
        "Column": silver_master_df.columns,
        "Missing Values": missing_count.values,
        "Missing %": missing_percent.values
    })
    print("\n===== 2. MISSING VALUE PROFILE =====")
    display(missing_profile)
    
except Exception as e:
    print("Missing value profiling failed")
    print(e)
    raise
# =========================================================
# 4. DUPLICATE PROFILING
# =========================================================
# Check for duplicate records in dataset
try:
    duplicate_rows = silver_master_df.duplicated().sum()
    
    duplicate_profile = pd.DataFrame({
        "Metric": ["Duplicate Rows"],
        "Value": [duplicate_rows]
    })
    
    print("\n===== 3. DUPLICATE PROFILE =====")
    display(duplicate_profile)
    
except Exception as e:
    print("Duplicate records count profiling failed")
    print(e)
    raise
# =========================================================
# 5. DATA TYPE PROFILING
# =========================================================
# Validate column data types for consistency
try:
    dtype_profile = pd.DataFrame({
        "Column": silver_master_df.columns,
        "Data Type": silver_master_df.dtypes.astype(str).values
    })
    
    print("\n===== 4. DATA TYPE PROFILE =====")
    display(dtype_profile)
except Exception as e:
    print("Data type profiling failed")
    print(e)
    raise
# =========================================================
# 6. UNIQUE VALUE PROFILING
# =========================================================
# Identify cardinality (unique values) of each column
try:
    unique_profile = pd.DataFrame({
        "Column": silver_master_df.columns,
        "Unique Values": silver_master_df.nunique().values
    })
    
    print("\n===== 5. UNIQUE VALUE PROFILE =====")
    display(unique_profile)
    
except Exception as e:
    print("Data type profiling failed")
    print(e)
    raise
# =========================================================
# 7. NUMERIC STATISTICAL PROFILING
# =========================================================
# Generate descriptive statistics for numeric columns

print("\n===== 6. NUMERIC STATISTICS =====")
display(silver_master_df.describe())

# =========================================================
# 8. SILVER LAYER DATA CLEANING (IMPROVED)
# =========================================================

try:
    silver_df = silver_master_df.copy()

    key_columns = ["month", "town", "flat_type", "resale_price"]

    # -----------------------------------------------------
    # 1. Identify missing values (instead of immediately dropping)
    # -----------------------------------------------------
    silver_df["is_missing_key"] = silver_df[key_columns].isnull().any(axis=1)

    missing_count = silver_df["is_missing_key"].sum()
    print("Missing key records:", missing_count)

    # -----------------------------------------------------
    # 2. Handle missing values
    # -----------------------------------------------------

    # Fill categorical columns
    silver_df["town"] = silver_df["town"].fillna("UNKNOWN")
    silver_df["flat_type"] = silver_df["flat_type"].fillna("UNKNOWN")

    # Drop rows where resale_price or month is missing (critical fields)
    silver_df = silver_df.dropna(subset=["resale_price", "month"])

    # Ensure no duplicates
    silver_df = silver_df.drop_duplicates()

    print("\n===== SILVER LAYER SAMPLE DATA =====")
    display(silver_df.head(5))

    print("Final rows:", len(silver_df))

except Exception as e:
    print("Silver Data Cleaning failed")
    print(e)
    raise
# =========================================================
# 9. Outlier Detection
# Simple statistical checks can highlight unusual resale prices:
# =========================================================
try:
    outlier_detection_q1 = silver_df["resale_price"].quantile(0.25)
    outlier_detection_q3 = silver_df["resale_price"].quantile(0.75)
    iqr = outlier_detection_q3 - outlier_detection_q1
    outliers = silver_df[(silver_df["resale_price"] < (outlier_detection_q1 - 1.5*iqr)) | (silver_df["resale_price"] > (outlier_detection_q3 + 1.5*iqr))]
    
    print("\n===== 8. OUTLIER DETECTION SUMMARY =====")
    print("Outlier Count:", len(outliers))
    display(outliers.head(5))
    
except Exception as e:
    print("Error in outlier detection")
    print(e)
    raise
# =========================================================
# 10. Categorical Profiling 
# =========================================================
print("\n===== 9. CATEGORICAL PROFILING =====")
print(silver_df["flat_type"].value_counts())

# =========================================================
# 11. Numeric Consistency Checks   
# =========================================================
print("\n===== 10. NUMERIC CONSISTENCY CHECKS =====")
print("Resale Price Range:", silver_df["resale_price"].min(), "-", silver_df["resale_price"].max())
print("Floor Area Range:", silver_df["floor_area_sqm"].min(), "-", silver_df["floor_area_sqm"].max())

# =========================================================
# 12. Date Validation   
# =========================================================
try:
    silver_df["month"] = pd.to_datetime(silver_df["month"], format="%Y-%m", errors="coerce")
    invalid_dates = silver_df[silver_df["month"].isnull()]
    print("\n===== 11. DATE VALIDATION =====")
    print("Invalid date rows:", len(invalid_dates))
    
except Exception as e:
    print("Error in date conversion")
    print(e)
    raise
# =========================================================
# 13. Extended Categorical Profiling   
# Add distributions for storey_range and flat_model:
# =========================================================
print("\n===== 12. STOREY RANGE DISTRIBUTION =====")
display(silver_df["storey_range"].value_counts())

# =========================================================
# 14. Correlation Analysis     
# Check relationships between numeric variables:
# =========================================================
print("\n===== 13. CORRELATION MATRIX =====")
print(silver_df[["floor_area_sqm", "resale_price"]].corr())

# =========================================================
# 15. Outlier Reporting     
# Instead of just displaying outliers, summarize them:
# =========================================================
print("\n===== 14. OUTLIER REPORTING =====")
print("Outlier resale price min:", outliers["resale_price"].min())
print("Outlier resale price max:", outliers["resale_price"].max())

# =========================================================
# 16. FINAL DATA VALIDATION
# =========================================================
# Compare dataset size before and after cleaning

summary_table = pd.DataFrame({
    "Metric": ["Total Rows (Raw)", "Duplicate Rows", "Final Silver Rows"],
    "Value": [
        len(silver_master_df),
        duplicate_rows,
        len(silver_df)
    ]
})

print("\n===== 15. FINAL SUMMARY =====")
print(summary_table)

# =========================================================
# 17. SAVE SILVER LAYER DATASET
# =========================================================
# Export cleaned dataset to Silver layer storage path

try:
   
    silver_df.to_csv(Q2_Sliver_Output_Path, index=False)
    
    print("\n===== SILVER LAYER PROCESS COMPLETED SUCCESSFULLY =====")
    print("Output saved at:", Q2_Sliver_Output_Path)

    # =========================================================
    # 18. FINAL SILVER DATA PREVIEW
    # =========================================================
    # Display dataset sample for validation
    
    print("\n===== FINAL SILVER DATA SAMPLE =====")
    display(silver_df.head(5))
    
except Exception as e:
    print("Error saving file")
    print(e)
    raise


===== SILVER LAYER DATA PROFILING STARTED =====

===== 1. BASIC DATA OVERVIEW =====

Total Records: 92544

 ---- Dataset Info: ---- 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92544 entries, 0 to 92543
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                92544 non-null  object 
 1   town                 92544 non-null  object 
 2   flat_type            92544 non-null  object 
 3   block                92544 non-null  object 
 4   street_name          92544 non-null  object 
 5   storey_range         92544 non-null  object 
 6   floor_area_sqm       92544 non-null  float64
 7   flat_model           92544 non-null  object 
 8   lease_commence_date  92544 non-null  int32  
 9   remaining_lease      37153 non-null  float64
 10  resale_price         92544 non-null  float64
dtypes: float64(3), int32(1), object(7)
memory usage: 7.4+ MB
None

 ---- SAMPLE DATA ---- 


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2012-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,NaN,257800.0
1,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,NaN,263000.0
2,2012-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,NaN,275000.0
3,2012-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,NaN,260000.0
4,2012-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,NaN,226000.0



===== 2. MISSING VALUE PROFILE =====


,Column,Missing Values,Missing %
0,month,0,0.000000
1,town,0,0.000000
2,flat_type,0,0.000000
3,block,0,0.000000
4,street_name,0,0.000000
5,storey_range,0,0.000000
6,floor_area_sqm,0,0.000000
7,flat_model,0,0.000000
8,lease_commence_date,0,0.000000
9,remaining_lease,55391,59.853691



===== 3. DUPLICATE PROFILE =====


...

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,is_missing_key
205,2012-01,BISHAN,EXECUTIVE,134,BISHAN ST 12,01 TO 03,151.0,Apartment,1986,NaN,765000.0,False
308,2012-01,BUKIT MERAH,5 ROOM,2C,BOON TIONG RD,13 TO 15,115.0,Improved,2001,NaN,838000.0,False
315,2012-01,BUKIT MERAH,5 ROOM,12,JLN BT MERAH,22 TO 24,127.0,Improved,1996,NaN,790000.0,False
316,2012-01,BUKIT MERAH,5 ROOM,23,JLN MEMBINA,13 TO 15,110.0,Improved,2003,NaN,840000.0,False
318,2012-01,BUKIT MERAH,5 ROOM,45,LENGKOK BAHRU,01 TO 03,139.0,Model A-Maisonette,1984,NaN,870000.0,False



===== 9. CATEGORICAL PROFILING =====
flat_type
4 ROOM              36441
3 ROOM              26198
5 ROOM              21309
EXECUTIVE            7288
2 ROOM                952
1 ROOM                 56
MULTI-GENERATION       27
Name: count, dtype: int64

===== 10. NUMERIC CONSISTENCY CHECKS =====
Resale Price Range: 190000.0 - 1150000.0
Floor Area Range: 31.0 - 280.0

===== 11. DATE VALIDATION =====
Invalid date rows: 0

===== 12. STOREY RANGE DISTRIBUTION =====


storey_range
04 TO 06    21135
07 TO 09    18707
01 TO 03    17414
10 TO 12    15985
13 TO 15     6684
16 TO 18     2697
01 TO 05     2696
06 TO 10     2472
11 TO 15     1259
19 TO 21     1152
22 TO 24      745
25 TO 27      380
16 TO 20      265
28 TO 30      236
21 TO 25       92
34 TO 36       88
37 TO 39       81
31 TO 33       79
26 TO 30       39
40 TO 42       38
46 TO 48        8
43 TO 45        8
36 TO 40        7
31 TO 35        2
49 TO 51        2
Name: count, dtype: int64


===== 13. CORRELATION MATRIX =====
                floor_area_sqm  resale_price
floor_area_sqm        1.000000      0.700674
resale_price          0.700674      1.000000

===== 14. OUTLIER REPORTING =====
Outlier resale price min: 752500.0
Outlier resale price max: 1150000.0

===== 15. FINAL SUMMARY =====
              Metric  Value
0   Total Rows (Raw)  92544
1     Duplicate Rows    273
2  Final Silver Rows  92271

===== SILVER LAYER PROCESS COMPLETED SUCCESSFULLY =====
Output saved at: C:/Users/ilang/HDB/hdb_resale/Q2_HDB_Silver_Level_Data/Master_Dataset.csv

===== FINAL SILVER DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,is_missing_key
0,2012-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,NaN,257800.0,False
1,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,NaN,263000.0,False
2,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,NaN,275000.0,False
3,2012-01-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,NaN,260000.0,False
4,2012-01-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,NaN,226000.0,False


In [3]:
'''
Q3. Design data validation rules to validate the following fields (i.e. Date, Town, Flat Type, Flat Model, storey_range) based on the statistical 
properties of this master dataset. 
'''

"""
Task Name: HDB Data Validation Rules
Created By: ILANGO RAMESH
Created Date: 08-Apr-2026

Purpose:
--------
Validate key fields (Date, Town, Flat Type, Flat Model, Storey Range) in the HDB master dataset 
based on their statistical properties. The script standardizes values, applies validation rules, 
and separates valid records from invalid ones for further processing.
"""

# Load dataset
try:
    # data_validation_rules_df = pd.read_csv(Q2_Sliver_Output_Path)
    data_validation_rules_df = silver_df.copy()
except Exception as e:
    print("Error loading dataset")
    print(e)
    raise


# =========================================================
# 1. DATE FIELD VALIDATION (MONTH)
# =========================================================

"""
1. Date Field Validation (month)
"""

try:
    # CLEAN FIRST
    data_validation_rules_df["month"] = (data_validation_rules_df["month"].astype(str).str.strip())

    def validate_month(x):
        try:
            x = str(x).strip()

            # YYYY-MM
            if "-" in x and len(x.split("-")) == 2:
                year, month = x.split("-")

            # YYYY/MM
            elif "/" in x:
                year, month = x.split("/")

            # YYYY-MM-DD
            elif len(x.split("-")) == 3:
                year, month, _ = x.split("-")

            else:
                return False

            year = int(year)
            month = int(month)

            return 1990 <= year <= 2100 and 1 <= month <= 12

        except:
            return False

    data_validation_rules_df["is_valid_month"] = data_validation_rules_df["month"].apply(validate_month)
    # print("Date Field validation done")

except Exception as e:
    print("Error in month validation")
    print(e)
    raise


# =========================================================
# 2. TOWN FIELD VALIDATION
# =========================================================
try:
    valid_towns = ["Ang Mo Kio", "Bedok", "Bishan", "Bukit Batok", "Bukit Merah", "Bukit Panjang",
                   "Bukit Timah", "Central Area", "Choa Chu Kang", "Clementi", "Geylang", "Hougang",
                   "Jurong East", "Jurong West", "Kallang/Whampoa", "Marine Parade", "Pasir Ris",
                   "Punggol", "Queenstown", "Sembawang", "Sengkang", "Serangoon", "Tampines",
                   "Toa Payoh", "Woodlands", "Yishun"]
    
    data_validation_rules_df["town_clean"] = (
        data_validation_rules_df["town"]
        .astype(str)
        .str.strip()
        .str.title()
    )
    
    data_validation_rules_df["is_valid_town"] = (
        data_validation_rules_df["town_clean"]
        .isin(valid_towns)
        .fillna(False)
    )
    # print("Town Field validation done")
except Exception as e:
    print("Error in town field validation")
    print(e)
    raise

# =========================================================
# 3. FLAT TYPE VALIDATION
# =========================================================

try:
    valid_flat_types = ["1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"]
    
    data_validation_rules_df["flat_type_clean"] = (
        data_validation_rules_df["flat_type"]
        .astype(str)
        .str.upper()
        .str.strip()
    )
    
    data_validation_rules_df["is_valid_flat_type"] = (
        data_validation_rules_df["flat_type_clean"]
        .isin(valid_flat_types)
        .fillna(False)
    )

   #  print("Flat Type validation done")
except Exception as e:
    print("Error in flat type validation")
    print(e)
    raise
# =========================================================
# 4. FLAT MODEL VALIDATION
# =========================================================
try:
    flat_model_mapping = {
        "Model A-Maisonette": "Maisonette",
        "Improved-Maisonette": "Maisonette",
        "Premium Apartment Loft": "Apartment",
        "Premium Apartment": "Apartment",
        "Premium Maisonette": "Maisonette",
        "Model A2": "Model A",
        "Adjoined flat": "Apartment",
        "Multi Generation": "Maisonette",
        "DBSS": "Apartment",
        "Type S1": "Standard",
        "Type S2": "Standard",
        "2-room": "2 Room"
    }
    
    data_validation_rules_df["flat_model_clean"] = (
        data_validation_rules_df["flat_model"]
        .replace(flat_model_mapping)
        .astype(str)
        .str.strip()
        .str.title()
    )
    
    valid_flat_models = [
        "Improved", "New Generation", "Model A", "Standard", "Simplified", "Premium",
        "Maisonette", "Apartment", "Terrace", "2 Room"
    ]
    
    data_validation_rules_df["is_valid_flat_model"] = (
        data_validation_rules_df["flat_model_clean"]
        .isin(valid_flat_models)
        .fillna(False)
    )
   #  print("Flat Model validation done")
except Exception as e:
    print("Error in flat model validation")
    print(e)
    raise

# =========================================================
# 5. STOREY RANGE VALIDATION (FIXED)
# =========================================================

try:
    data_validation_rules_df["storey_range_clean"] = (
        data_validation_rules_df["storey_range"]
        .astype(str)
        .str.upper()
        .str.replace("-", " TO ")
        .str.strip()
    )

    def validate_storey(x):
        try:
            parts = x.split("TO")
            low = int(parts[0].strip())
            high = int(parts[1].strip())
            return low < high
        except:
            return False

    data_validation_rules_df["is_valid_storey_range"] = (
        data_validation_rules_df["storey_range_clean"]
        .apply(validate_storey)
    )
   #  print("Flat storey range validation done")
except Exception as e:
    print("Error in storey range validation")
    print(e)
    raise


# =========================================================
# 6. FINAL VALID/INVALID SPLIT
# =========================================================
try:
    validation_cols = [
        "is_valid_month",
        "is_valid_town",
        "is_valid_flat_type",
        "is_valid_flat_model",
        "is_valid_storey_range"
    ]
    
    valid_df = data_validation_rules_df[data_validation_rules_df[validation_cols].all(axis=1)].copy()
    invalid_df = data_validation_rules_df[~data_validation_rules_df[validation_cols].all(axis=1)].copy()
    
    valid_df.to_csv(Q3_Valid_Dataset_Output_Path, index=False)
    invalid_df.to_csv(Q3_Invalid_Dataset_Output_Path, index=False)
    
   # Summary report    
    print("Valid records:", len(valid_df))
    print("Invalid records:", len(invalid_df))
    print("Invalid counts per field:")
    print(invalid_df[validation_cols].apply(lambda x: (~x).sum()))
    
    print("Files saved successfully")
    
    # =========================================================
    # 7. FINAL SILVER DATA PREVIEW
    # =========================================================
    # Display dataset sample for validation
    print("\n===== FINAL SILVER DATA SAMPLE =====")
    display(valid_df.head(5))

except Exception as e:
    print("Error saving files")
    print(e)
    raise




Valid records: 92271
Invalid records: 0
Invalid counts per field:
is_valid_month           0.0
is_valid_town            0.0
is_valid_flat_type       0.0
is_valid_flat_model      0.0
is_valid_storey_range    0.0
dtype: float64
Files saved successfully

===== FINAL SILVER DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,is_missing_key,is_valid_month,town_clean,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range
0,2012-01-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,01 TO 03,44.0,Improved,1979,NaN,...,False,True,Ang Mo Kio,True,2 ROOM,True,Improved,True,01 TO 03,True
1,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,07 TO 09,44.0,Improved,1978,NaN,...,False,True,Ang Mo Kio,True,2 ROOM,True,Improved,True,07 TO 09,True
2,2012-01-01,ANG MO KIO,2 ROOM,314,ANG MO KIO AVE 3,10 TO 12,44.0,Improved,1978,NaN,...,False,True,Ang Mo Kio,True,2 ROOM,True,Improved,True,10 TO 12,True
3,2012-01-01,ANG MO KIO,2 ROOM,170,ANG MO KIO AVE 4,01 TO 03,45.0,Improved,1986,NaN,...,False,True,Ang Mo Kio,True,2 ROOM,True,Improved,True,01 TO 03,True
4,2012-01-01,ANG MO KIO,2 ROOM,174,ANG MO KIO AVE 4,07 TO 09,45.0,Improved,1986,NaN,...,False,True,Ang Mo Kio,True,2 ROOM,True,Improved,True,07 TO 09,True


In [4]:
'''
4. Assume HDB lease is 99 years old, recompute remaining lease as of today. Remaining lease should be rounded down to Years and Months. 
'''

"""
Task Name: HDB Lease Recalculation
Created By: ILANGO RAMESH
Created Date: 08-Apr-2026

Purpose: 
--------
This script loads the cleaned HDB master dataset and recomputes the remaining lease 
for each flat, assuming a 99-year lease period. The remaining lease is calculated 
as of today's date and expressed in years and months (rounded down).
"""
try:
    # Load dataset
   # remaining_lease_df = pd.read_csv(Q3_Valid_Dataset_Output_Path)
    remaining_lease_df =  valid_df.copy()
except Exception as e:
    print("Error loading dataset")
    print(e)
    raise
    
# Current date for lease calculation
today = datetime.today()

# Function to compute remaining lease
def compute_remaining_lease(lease_year):
    try:
        """
        Computes the remaining lease given the lease commencement year.
        Assumes a 99-year lease period.
        Returns the remaining lease in 'X years Y months' format.
        """
        if pd.isnull(lease_year):
            return None

        lease_start = datetime(int(lease_year), 1, 1)
        lease_end = lease_start.replace(year=lease_start.year + 99)

        delta = lease_end - datetime.today()

        years = delta.days // 365
        months = (delta.days % 365) // 30

        return f"{years} years {months} months"

    except Exception as e:
        print(f"Error processing lease_year: {lease_year}")
        print(e)
        return None

try:
    # Apply function to dataset
    remaining_lease_df["remaining_lease_recomputed"] = remaining_lease_df["lease_commence_date"].apply(compute_remaining_lease)
    print("Lease computation completed")

except Exception as e:
    print("Error during lease computation")
    print(e)
    raise

try:
    # Save updated dataset with recomputed lease
    remaining_lease_df.to_csv(Q4_Lease_Output_Path)
    
    # Quick check of results
    #print(remaining_lease_df[["lease_commence_date", "remaining_lease_recomputed"]])     
    # Display final cleaned dataset sample for validation

    
    print("File saved successfully:", Q4_Lease_Output_Path)
    
     # Display dataset sample for validation
    print("\n===== FINAL LEASE COMPUTATION DATA SAMPLE =====")
    display(remaining_lease_df[["month","town" ,"block","street_name","storey_range","flat_model", "lease_commence_date", "remaining_lease_recomputed"]].head(5))
    #display(remaining_lease_df.head(10))

except Exception as e:
    print("Error saving file")
    print(e)
    raise

Lease computation completed
File saved successfully: C:/Users/ilang/HDB/hdb_resale/Q4_HDB_Lease_Recalculation/Master_Dataset_With_Lease.csv

===== FINAL LEASE COMPUTATION DATA SAMPLE =====


,month,town,block,street_name,storey_range,flat_model,lease_commence_date,remaining_lease_recomputed
0,2012-01-01,ANG MO KIO,406,ANG MO KIO AVE 10,01 TO 03,Improved,1979,51 years 9 months
1,2012-01-01,ANG MO KIO,314,ANG MO KIO AVE 3,07 TO 09,Improved,1978,50 years 9 months
2,2012-01-01,ANG MO KIO,314,ANG MO KIO AVE 3,10 TO 12,Improved,1978,50 years 9 months
3,2012-01-01,ANG MO KIO,170,ANG MO KIO AVE 4,01 TO 03,Improved,1986,58 years 9 months
4,2012-01-01,ANG MO KIO,174,ANG MO KIO AVE 4,07 TO 09,Improved,1986,58 years 9 months


In [5]:
'''
5. You may assume the composite key for the dataset is all columns except the resale price. If 
all columns have the exact same value, except for the resale price (i.e. duplicated key), take 
the higher price and discard the lower price into the “failed” dataset (if any). 
'''

"""
Task Name: HDB Composite Key Deduplication
Created By: ILANGO RAMESH
Created Date: 08-Apr-2026

Purpose:
--------
Ensure each record is unique based on all columns except resale_price.
If duplicates exist, keep the record with the highest resale price and move the lower-priced duplicates to a failed dataset.
"""
try:
    # Load dataset with lease information
    # hdb_composite_key_deduplication_df = pd.read_csv(Q4_Lease_Output_Path)
    hdb_composite_key_deduplication_df = remaining_lease_df.copy()
    
except Exception as e:
    print("Error loading dataset")
    print(e)
    raise
    
#hdb_composite_key_deduplicati_df = remaining_lease_df
try:   
    # Composite key = all columns except resale_price
    composite_key = [col for col in hdb_composite_key_deduplication_df.columns if col != "resale_price"]
    
    # Sort dataset by resale_price descending (highest first)
    sorted_df = hdb_composite_key_deduplication_df.sort_values(by="resale_price", ascending=False)
    
    # Drop duplicates based on composite key, keeping the highest resale price
    unique_df = sorted_df.drop_duplicates(subset=composite_key, keep="first")
    #print("Deduplication completed")

except Exception as e:
    print("Error during deduplication logic")
    print(e)
    raise
    
try:    
    # Identify failed records (lower-priced duplicates)
    failed_df = sorted_df[~sorted_df.index.isin(unique_df.index)]
   # print("Failed dataset created")

except Exception as e:
    print("Error creating failed dataset")
    print(e)
    raise
try:
    # Save outputs
    unique_df.to_csv(Q5_Compo_Dedup_Unique_Dataset_Output_Path)
    failed_df.to_csv(Q5_Compo_Dedup_Failed_Dataset_Output_Path)
    
    # Summary report
    print("Final dataset records:", len(unique_df))
    print("Failed dataset records:", len(failed_df))

    print("Files saved successfully")

    # Display dataset sample for validation
    print("\n===== COMPOSITE KEY DEUPLICATION UNIQUE DATA SAMPLE =====")
    display(unique_df.head(5))
     
    print("\n===== COMPOSITE KEY DEUPLICATION FAILED DATA SAMPLE =====")
    display(failed_df.head(5))

except Exception as e:
    print("Error saving files")
    print(e)
    raise
   

Final dataset records: 90949
Failed dataset records: 1322
Files saved successfully

===== COMPOSITE KEY DEUPLICATION UNIQUE DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,is_valid_month,town_clean,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,remaining_lease_recomputed
91881,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,54.0,...,True,Kallang/Whampoa,True,3 ROOM,True,Terrace,True,01 TO 03,True,44 years 9 months
86699,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,Type S2,2011,93.0,...,True,Central Area,True,5 ROOM,True,Standard,True,43 TO 45,True,83 years 9 months
89988,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,Type S2,2011,93.0,...,True,Central Area,True,5 ROOM,True,Standard,True,31 TO 33,True,83 years 9 months
89989,2016-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.0,Type S2,2011,93.0,...,True,Central Area,True,5 ROOM,True,Standard,True,46 TO 48,True,83 years 9 months
85402,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,93.0,...,True,Kallang/Whampoa,True,5 ROOM,True,Apartment,True,28 TO 30,True,83 years 9 months



===== COMPOSITE KEY DEUPLICATION FAILED DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,is_valid_month,town_clean,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,remaining_lease_recomputed
31982,2013-07-01,BUKIT TIMAH,EXECUTIVE,7,TOH YI DR,04 TO 06,146.0,Maisonette,1989,NaN,...,True,Bukit Timah,True,EXECUTIVE,True,Maisonette,True,04 TO 06,True,61 years 9 months
70687,2015-11-01,CENTRAL AREA,4 ROOM,1A,CANTONMENT RD,37 TO 39,95.0,Type S1,2011,94.0,...,True,Central Area,True,4 ROOM,True,Standard,True,37 TO 39,True,83 years 9 months
88149,2016-10-01,BISHAN,5 ROOM,273B,BISHAN ST 24,01 TO 03,120.0,DBSS,2011,93.0,...,True,Bishan,True,5 ROOM,True,Apartment,True,01 TO 03,True,83 years 9 months
31822,2013-07-01,BISHAN,EXECUTIVE,187,BISHAN ST 13,07 TO 09,146.0,Maisonette,1987,NaN,...,True,Bishan,True,EXECUTIVE,True,Maisonette,True,07 TO 09,True,59 years 9 months
56937,2015-02-01,CENTRAL AREA,4 ROOM,1D,CANTONMENT RD,37 TO 39,95.0,Type S1,2011,94.0,...,True,Central Area,True,4 ROOM,True,Standard,True,37 TO 39,True,83 years 9 months


In [6]:
'''
6. Identify any potentially anomalous resale price based on any heuristics you deem 
appropriate. Please document your heuristic and assumptions in your documentations.  
'''

"""
Task Name: HDB Resale Price Anomaly Detection
Created By: ILANGO RAMESH
Created Date: 08-Apr-2026

Purpose:
--------
Identify potentially anomalous resale prices in the HDB dataset using statistical heuristics.
Assumption: Resale prices within each flat type and town follow a normal distribution.
Heuristic: Use the Interquartile Range (IQR) method to flag extreme outliers.
"""

try:
    # Load dataset
    hdb_resaleprice_anomalydetection_df = unique_df.copy()
except Exception as e:
    print("Error loading dataset")
    print(e)
    raise

# =========================================================
# FIX: FUNCTION MUST BE OUTSIDE TRY BLOCK
# =========================================================

def flag_anomalies(group):
    Q1 = group["resale_price"].quantile(0.25) # 1st Quartile
    Q3 = group["resale_price"].quantile(0.75) # 3rd Quartile
    
    #Interquartile Range
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    group["is_anomaly"] = ~group["resale_price"].between(lower_bound, upper_bound)
    return group

try:
    # Apply anomaly detection per flat type and town
    hdb_resaleprice_anomalydetection_df = (
        hdb_resaleprice_anomalydetection_df
        .groupby(["flat_type_clean", "town_clean"], group_keys=False)
        .apply(flag_anomalies)
        .reset_index(drop=True)
    )

except Exception as e:
    print("Error during anomaly detection")
    print(e)
    raise


try:
    # Separate anomaly records
    anomaly_df = hdb_resaleprice_anomalydetection_df[
        hdb_resaleprice_anomalydetection_df["is_anomaly"]
    ]

    normal_df = hdb_resaleprice_anomalydetection_df[
        ~hdb_resaleprice_anomalydetection_df["is_anomaly"]
    ]

    # Save outputs
    anomaly_df.to_csv(Q6_Price_Anomaly_Detection_Output_Path, index=False)
    normal_df.to_csv(Q6_Price_Normal_Detection_Output_Path, index=False)

    # Summary
    print("Total records:", len(hdb_resaleprice_anomalydetection_df))
    print("Anomalous records:", len(anomaly_df))
    print("Normal records:", len(normal_df))

    print("Files saved successfully")

    # Display dataset sample for validation
    print("\n===== ANOMALY DATA SAMPLE =====")
    display(anomaly_df.head(5))

    print("\n===== NORMAL DATA SAMPLE =====")
    display(normal_df.head(5))

except Exception as e:
    print("Error saving files")
    print(e)
    raise

C:\Users\ilang\AppData\Local\Temp\ipykernel_22660\1390724391.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(flag_anomalies)


Total records: 90949
Anomalous records: 1307
Normal records: 89642
Files saved successfully

===== ANOMALY DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,town_clean,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,remaining_lease_recomputed,is_anomaly
0,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,54.0,...,Kallang/Whampoa,True,3 ROOM,True,Terrace,True,01 TO 03,True,44 years 9 months,True
4,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,93.0,...,Kallang/Whampoa,True,5 ROOM,True,Apartment,True,28 TO 30,True,83 years 9 months,True
5,2014-10-01,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.0,Maisonette,1987,NaN,...,Bishan,True,EXECUTIVE,True,Maisonette,True,22 TO 24,True,59 years 9 months,True
12,2015-03-01,KALLANG/WHAMPOA,3 ROOM,53,JLN MA'MOR,01 TO 03,280.0,Terrace,1972,56.0,...,Kallang/Whampoa,True,3 ROOM,True,Terrace,True,01 TO 03,True,44 years 9 months,True
14,2013-11-01,BISHAN,EXECUTIVE,190,BISHAN ST 13,19 TO 21,150.0,Maisonette,1987,NaN,...,Bishan,True,EXECUTIVE,True,Maisonette,True,19 TO 21,True,59 years 9 months,True



===== NORMAL DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,town_clean,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,remaining_lease_recomputed,is_anomaly
1,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,Type S2,2011,93.0,...,Central Area,True,5 ROOM,True,Standard,True,43 TO 45,True,83 years 9 months,False
2,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,Type S2,2011,93.0,...,Central Area,True,5 ROOM,True,Standard,True,31 TO 33,True,83 years 9 months,False
3,2016-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.0,Type S2,2011,93.0,...,Central Area,True,5 ROOM,True,Standard,True,46 TO 48,True,83 years 9 months,False
6,2015-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,43 TO 45,107.0,Type S2,2011,94.0,...,Central Area,True,5 ROOM,True,Standard,True,43 TO 45,True,83 years 9 months,False
7,2016-08-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,105.0,Type S2,2011,93.0,...,Central Area,True,5 ROOM,True,Standard,True,31 TO 33,True,83 years 9 months,False


In [7]:
'''
7. Any additional data cleaning/validation rules that you may deem appropriate. The actual dataset may not have anomalous data, but mechanisms to identify such data can be 
demonstrated in your assignment.  

Data Transformation Requirements:  

1. Taking the cleaned data from the previous step, create a new column called “Resale 
Identifier”. The “Resale Identifier” is derived from:  - First character is “S”.  - Next 3 digits is the first 3 digits of the block column, after removing any characters. In 
the event the block column has less than 3 digits, prepend with the appropriate number 
of zeros in front. (E.g. if block number is “19”, digits will be “019”)  - Next 2 digits is derived by: Taking the 1st and 2nd digit of the average resale price, group 
by year-month, town and flat_type. For instance, if the average Ang Mo Kio, 2 Room 
Flats in Jan 2017 is $230000, the 2 digits is “23”. - The last two digits is the month of the current entry (e.g. if the month is 2012-01, the 
last two digits is 01).  - The final character is the first character of the town. E.g. Ang Mo Kio, the character is 
“A”. 
2. If there are any duplicate records, take the higher price and discard the lower price one. 
3. Hash this identifier column using an irreversible hashing algorithm, while preserving its 
uniqueness. Explain the hashing algorithm that you adopted in your documentations.  

'''


"""
Script Name: HDB Data Transformation
Created By: ILANGO RAMESH
Created Date: 09-Apr-2026

Purpose:
--------
Perform additional data cleaning and transformation on HDB dataset.
Includes:
1. Resale Identifier creation
2. Duplicate handling
3. Hashing identifier (SHA-256)
"""
# =========================
# LOAD DATA
# =========================
try:
    hdb_data_transformation_df = valid_df.copy()
except Exception as e:
    print("Error loading dataset")
    raise e

# =========================
# CREATE YEAR-MONTH COLUMN
# =========================
hdb_data_transformation_df["year_month"] = (
    pd.to_datetime(hdb_data_transformation_df["month"])
    .dt.to_period("M")
    .astype(str)
)


# =========================
# COMPUTE AVG RESALE PRICE
# =========================
try:
    avg_prices = (
        hdb_data_transformation_df
        .groupby(["year_month", "town", "flat_type"])["resale_price"]
        .mean()
        .round(2)
        .reset_index()
        .rename(columns={"resale_price": "avg_resale_price"})
    )
except Exception as e:
    print("Error computing average resale price")
    raise e


# =========================
# MERGE AVG PRICE BACK
# =========================
try:
    hdb_data_transformation_df = hdb_data_transformation_df.merge(
        avg_prices,
        on=["year_month", "town", "flat_type"],
        how="left"
    )
except Exception as e:
    print("Error merging avg prices")
    raise e


# =========================
# CREATE RESALE IDENTIFIER
# =========================
def create_resale_identifier(row):
    identifier = "S"

    # ---- Block (3 digits) ----
    block_digits = ''.join(filter(str.isdigit, str(row["block"] or "")))
    block_digits = block_digits.zfill(3)[-3:]
    identifier += block_digits

    # ---- Avg price (first 2 digits) ----
    avg_price = int(row["avg_resale_price"])
    identifier += str(avg_price)[:2]

    # ---- Month (MM) ----
    month_digits = pd.to_datetime(row["year_month"]).month
    identifier += f"{month_digits:02d}"

    # ---- Town initial ----
    identifier += str(row["town"])[0].upper()

    return identifier


try:
    hdb_data_transformation_df["Resale_Identifier"] = (
        hdb_data_transformation_df.apply(create_resale_identifier, axis=1)
    )
except Exception as e:
    print("Error creating Resale Identifier")
    raise e


# =========================
# HANDLE DUPLICATES
# Keep highest resale price per identifier
# =========================
sorted_df = hdb_data_transformation_df.sort_values(
    by="resale_price",
    ascending=False
)

unique_df = sorted_df.drop_duplicates(
    subset=["Resale_Identifier"],
    keep="first"
)

failed_df = sorted_df[
    ~sorted_df.index.isin(unique_df.index)
]


# =========================
# HASH IDENTIFIER (SHA-256)
# =========================
unique_df["Resale_Identifier_Hashed"] = unique_df["Resale_Identifier"].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

hashed_df = unique_df.copy()


# =========================
# SAVE OUTPUTS
# =========================
try:
    unique_df.to_csv(Q7_hdb_data_transformed_Output_Path, index=False)
    failed_df.to_csv(Q7_hdb_data_failed_Output_Path, index=False)
    hashed_df.to_csv(Q7_hdb_data_hashed_Output_Path, index=False)

    print("Raw dataset records:", len(hdb_data_transformation_df))
    print("Transformed dataset records:", len(unique_df))
    print("Failed dataset records:", len(failed_df))
    print("Hashed dataset records:", len(hashed_df))
    print("Files saved successfully")
    
    # =========================
    # DISPLAY SAMPLES
    # =========================
    # Display dataset sample for validation
    print("\n===== TRANSFORMED DATA SAMPLE =====")
    display(unique_df.head())
    
    print("\n===== FAILED DATA SAMPLE =====")
    display(failed_df.head())
    
    print("\n===== HASHED DATA SAMPLE =====")
    display(hashed_df.head())
    
except Exception as e:
    print("Error saving files")
    raise e




C:\Users\ilang\AppData\Local\Temp\ipykernel_22660\2361871655.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unique_df["Resale_Identifier_Hashed"] = unique_df["Resale_Identifier"].apply(


Raw dataset records: 92271
Transformed dataset records: 77330
Failed dataset records: 14941
Hashed dataset records: 77330
Files saved successfully

===== TRANSFORMED DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,year_month,avg_resale_price,Resale_Identifier,Resale_Identifier_Hashed
91609,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,54.0,...,3 ROOM,True,Terrace,True,01 TO 03,True,2016-12,363380.95,S0573612K,689c73b5129f7c190b3583fadafe8ae5835d77ec5fb464...
86429,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,Type S2,2011,93.0,...,5 ROOM,True,Standard,True,43 TO 45,True,2016-09,1029600.00,S0011009C,dc3cb88029cf26ce1e87b42904bcd0b6eb033a6ebc5f80...
89717,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,Type S2,2011,93.0,...,5 ROOM,True,Standard,True,31 TO 33,True,2016-11,1028750.00,S0011011C,4dc0cb821bca135bb0aefdb588f218ea152babc0536de3...
85133,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,93.0,...,5 ROOM,True,Apartment,True,28 TO 30,True,2016-08,815636.36,S0088108K,86db259dfe16a112ef32c16be4093754a11a210f2187dc...
51129,2014-10-01,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.0,Maisonette,1987,NaN,...,EXECUTIVE,True,Maisonette,True,22 TO 24,True,2014-10,941944.00,S1949410B,71fde3746fa28628e4a78465c5741936b39995cc6df055...



===== FAILED DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,is_valid_town,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,year_month,avg_resale_price,Resale_Identifier
89718,2016-11-01,CENTRAL AREA,5 ROOM,1D,CANTONMENT RD,46 TO 48,107.0,Type S2,2011,93.0,...,True,5 ROOM,True,Standard,True,46 TO 48,True,2016-11,1028750.00,S0011011C
86428,2016-09-01,CENTRAL AREA,5 ROOM,1C,CANTONMENT RD,40 TO 42,106.0,Type S2,2011,93.0,...,True,5 ROOM,True,Standard,True,40 TO 42,True,2016-09,1029600.00,S0011009C
59309,2015-04-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,46 TO 48,107.0,Type S2,2011,94.0,...,True,5 ROOM,True,Standard,True,46 TO 48,True,2015-04,1019629.33,S0011004C
56696,2015-02-01,CENTRAL AREA,5 ROOM,1A,CANTONMENT RD,37 TO 39,107.0,Type S2,2011,94.0,...,True,5 ROOM,True,Standard,True,37 TO 39,True,2015-02,996166.67,S0019902C
73229,2016-01-01,CENTRAL AREA,5 ROOM,1F,CANTONMENT RD,28 TO 30,107.0,Type S2,2011,94.0,...,True,5 ROOM,True,Standard,True,28 TO 30,True,2016-01,949629.33,S0019401C



===== HASHED DATA SAMPLE =====


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,...,flat_type_clean,is_valid_flat_type,flat_model_clean,is_valid_flat_model,storey_range_clean,is_valid_storey_range,year_month,avg_resale_price,Resale_Identifier,Resale_Identifier_Hashed
91609,2016-12-01,KALLANG/WHAMPOA,3 ROOM,57,JLN MA'MOR,01 TO 03,259.0,Terrace,1972,54.0,...,3 ROOM,True,Terrace,True,01 TO 03,True,2016-12,363380.95,S0573612K,689c73b5129f7c190b3583fadafe8ae5835d77ec5fb464...
86429,2016-09-01,CENTRAL AREA,5 ROOM,1G,CANTONMENT RD,43 TO 45,106.0,Type S2,2011,93.0,...,5 ROOM,True,Standard,True,43 TO 45,True,2016-09,1029600.00,S0011009C,dc3cb88029cf26ce1e87b42904bcd0b6eb033a6ebc5f80...
89717,2016-11-01,CENTRAL AREA,5 ROOM,1B,CANTONMENT RD,31 TO 33,106.0,Type S2,2011,93.0,...,5 ROOM,True,Standard,True,31 TO 33,True,2016-11,1028750.00,S0011011C,4dc0cb821bca135bb0aefdb588f218ea152babc0536de3...
85133,2016-08-01,KALLANG/WHAMPOA,5 ROOM,8,BOON KENG RD,28 TO 30,119.0,DBSS,2011,93.0,...,5 ROOM,True,Apartment,True,28 TO 30,True,2016-08,815636.36,S0088108K,86db259dfe16a112ef32c16be4093754a11a210f2187dc...
51129,2014-10-01,BISHAN,EXECUTIVE,194,BISHAN ST 13,22 TO 24,150.0,Maisonette,1987,NaN,...,EXECUTIVE,True,Maisonette,True,22 TO 24,True,2014-10,941944.00,S1949410B,71fde3746fa28628e4a78465c5741936b39995cc6df055...
